In [17]:
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
import numpy as np
import KTreeModel


def create_cluster_graph(tree):
    centers, labels = tree.get_cluster_centers()
    labels = np.asarray(labels)
    pca = PCA(n_components=2)
    pca.fit(tree.train_x)

    plt.figure(figsize=(15,15), dpi=300)

    # variables
    mask = tree.train_y == 1
    x_true = tree.train_x[mask]
    mask = tree.train_y == 0
    x_false = tree.train_x[mask]
    #500 er örnek seç
    indices = np.random.choice(len(x_true), 500, replace=False)
    x_true = x_true[indices]
    indices = np.random.choice(len(x_false), 500, replace=False)
    x_false = x_false[indices]

    x_true = pca.transform(x_true)
    x_false = pca.transform(x_false)

    plt.scatter(x_true[:,0], x_true[:,1], c='green', marker='o', s = 5, edgecolors='none')
    plt.scatter(x_false[:,0], x_false[:,1], c='red', marker='o', s = 5, edgecolors='none')
    
    # centers
    x_2d = pca.transform(centers)

    idx_true = np.where(labels == 1)[0]
    if len(idx_true) > 50 :
        idx_true = np.random.choice(idx_true, 50, replace=False)

    idx_false = np.where(labels == 0)[0]
    if len(idx_false) > 50 :
        idx_false = np.random.choice(idx_false, 50, replace=False)

    idx = []
    for i in range(len(idx_true)):
        idx.append(idx_true[i])

    for i in range(len(idx_false)):
        idx.append(idx_false[i])


    colors = np.where(labels[idx] == 1, "green", "red")
    c = []
    c.extend(colors)
    plt.scatter(x_2d[idx,0], x_2d[idx,1], c=c, marker='s', s = 15, edgecolors='none')
    plt.title(f"k{tree.k}-d{tree.d}")

    import os
    os.makedirs("graphs", exist_ok= True)
    plt.savefig(f"graphs/k{tree.k}-d{tree.d}")
    plt.close()


import pandas as pd
csv_file = "results_final.csv"

df = pd.read_csv(csv_file)
df_sorted = df.sort_values("f1", ascending=False)
best = df_sorted[["k","d"]].head(1)

tree = KTreeModel.KTree()
for i, row in best.iterrows():
    
    tree.create_tree(int(row["k"]), int(row["d"]))
    cluster, label = tree.get_cluster_centers()
    create_cluster_graph(tree)

import os
#os.system("shutdown /s /t 0")

    
